# GridLock Hackathon 2.0 — Traffic Demand Prediction
## v7 — Lookup + Calibration (target: 96+)

**Key insight from diagnostics:**  
- 88.89% of test rows have a matching `(geohash, hour, minute)` in day 48 train  
- 96.35% match on `(geohash, hour)`  
- Day 49 demand systematically higher than day 48 (ratios 2.13/1.43/1.22 for h0/1/2, decaying)  
- Lookup × per-geohash-per-hour ratio gives **R² = 0.9225** on the *hardest* val (hours 0-2)  

**Strategy:** feed LightGBM `d48_lookup` (the baseline) + day-49 calibration features (the shift)  
and let it learn the residual.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stop, log_evaluation as lgb_log
import pygeohash as pgh

pd.set_option('display.max_columns', 80)
np.random.seed(42)
print('Imports OK')

## Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

for df in [train, test]:
    df['hour']   = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute'] = df['timestamp'].str.split(':').str[1].astype(int)

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Train days: {train["day"].value_counts().sort_index().to_dict()}')
print(f'Test  days: {test["day"].value_counts().sort_index().to_dict()}')

---
## Step 1: Standard Feature Engineering

In [ ]:
class FeatureEngineer:
    @staticmethod
    def _decode_gh(gh):
        try:
            lat, lng, _, _ = pgh.decode_exactly(gh)
            return float(lat), float(lng)
        except Exception:
            return 0.0, 0.0

    @staticmethod
    def _time_of_day(h):
        if   h <= 5:  return 0
        elif h <= 11: return 1
        elif h <= 17: return 2
        else:         return 3

    def fit(self, df):
        # Imputation
        self._temp_med_global = float(df['Temperature'].median())
        self._temp_med_gh     = df.groupby('geohash')['Temperature'].median()
        self._temp_mean_gh    = df.groupby('geohash')['Temperature'].mean()
        self._global_temp     = float(df['Temperature'].mean())

        # Geohash hierarchies
        self._gh_freq  = df['geohash'].value_counts().astype(float)
        gh4 = df['geohash'].str[:4]; gh5 = df['geohash'].str[:5]
        self._gh4_freq = gh4.value_counts().astype(float)
        self._gh5_freq = gh5.value_counts().astype(float)
        self._gh4_labels = {v: i for i, v in enumerate(sorted(gh4.unique()))}
        self._gh5_labels = {v: i for i, v in enumerate(sorted(gh5.unique()))}

        self.fitted = True
        return self

    def transform(self, df):
        df = df.copy()
        # Timestamp
        df['hour_sin']     = np.sin(2*np.pi*df['hour']/24)
        df['hour_cos']     = np.cos(2*np.pi*df['hour']/24)
        df['minute_sin']   = np.sin(2*np.pi*df['minute']/60)
        df['minute_cos']   = np.cos(2*np.pi*df['minute']/60)
        df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
        df['time_of_day']  = df['hour'].apply(self._time_of_day)

        # Geohash decode + hierarchy
        geo = df['geohash'].apply(self._decode_gh)
        df['geohash_lat'] = geo.apply(lambda x: x[0])
        df['geohash_lng'] = geo.apply(lambda x: x[1])
        df['geohash_4'] = df['geohash'].str[:4]
        df['geohash_5'] = df['geohash'].str[:5]
        df['geohash_4_label'] = df['geohash_4'].map(self._gh4_labels).fillna(-1).astype(int)
        df['geohash_5_label'] = df['geohash_5'].map(self._gh5_labels).fillna(-1).astype(int)
        df['geohash_frequency']   = df['geohash']  .map(self._gh_freq ).fillna(0.0)
        df['geohash_4_frequency'] = df['geohash_4'].map(self._gh4_freq).fillna(0.0)

        # Temperature
        df['Temperature'] = df['Temperature'].fillna(
            df['geohash'].map(self._temp_med_gh).fillna(self._temp_med_global))
        df['temp_deviation'] = df['Temperature'] - df['geohash'].map(
            self._temp_mean_gh).fillna(self._global_temp)

        # Categorical
        df['LargeVehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
        df['Landmarks']     = (df['Landmarks']     == 'Yes').astype(int)
        for rt in ['Residential', 'Street', 'Highway']:
            df[f'RoadType_{rt}'] = (df['RoadType'] == rt).astype(int)
        df['RoadType_Unknown'] = df['RoadType'].isna().astype(int)
        for wx in ['Sunny', 'Rainy', 'Foggy', 'Snowy']:
            df[f'Weather_{wx}'] = (df['Weather'] == wx).astype(int)
        df['Weather_Unknown'] = df['Weather'].isna().astype(int)

        return df

fe = FeatureEngineer().fit(train)
train_fe = fe.transform(train)
test_fe  = fe.transform(test)
print(f'Train FE: {train_fe.shape}  |  Test FE: {test_fe.shape}')

---
## Step 2: Day-48 Lookup Features (the baseline)

For each row, compute `d48_lookup_ghm` = day-48 mean demand at the same `(geohash, hour, minute)`.  
**Self-exclusion** for day-48 training rows: their own demand is excluded from the mean to prevent leakage.  
Fallback chain: `(g,h,m)` → `(g,h)` → `(g)` → global day-48 mean.  
This baseline alone explains the cross-geohash + intra-day variance.  
On day-49 val, this lookup gives R² ≈ 0.53 uncalibrated, but it's the building block.

In [ ]:
d48 = train_fe[train_fe['day'] == 48]
d48_global = float(d48['demand'].mean())

# Sum + count tables for day-48 demand at various granularities
ghm_sum  = d48.groupby(['geohash','hour','minute'])['demand'].sum()
ghm_cnt  = d48.groupby(['geohash','hour','minute'])['demand'].count()
gh_sum   = d48.groupby(['geohash','hour'])['demand'].sum()
gh_cnt   = d48.groupby(['geohash','hour'])['demand'].count()
g_sum    = d48.groupby('geohash')['demand'].sum()
g_cnt    = d48.groupby('geohash')['demand'].count()

def lookup_d48(df, exclude_self_day48=False):
    'Return d48_lookup_ghm, d48_lookup_gh, d48_lookup_g for each row.'
    n = len(df)
    gh_arr = df['geohash'].values
    h_arr  = df['hour'].values
    m_arr  = df['minute'].values
    day_arr = df['day'].values
    if exclude_self_day48 and 'demand' in df.columns:
        demand_arr = df['demand'].values
    else:
        demand_arr = np.zeros(n)

    ghm = np.empty(n); gh = np.empty(n); g = np.empty(n)
    for i in range(n):
        key3 = (gh_arr[i], h_arr[i], m_arr[i])
        key2 = (gh_arr[i], h_arr[i])
        key1 = gh_arr[i]
        # Self-exclusion only when this row is day 48 AND we asked for it
        excl = exclude_self_day48 and (day_arr[i] == 48)

        # (g,h,m) lookup
        if key3 in ghm_sum.index:
            s, c = ghm_sum.loc[key3], ghm_cnt.loc[key3]
            if excl:
                s -= demand_arr[i]; c -= 1
            if c > 0:
                ghm[i] = s / c
            elif key2 in gh_sum.index:
                ghm[i] = gh_sum.loc[key2] / gh_cnt.loc[key2]
            else:
                ghm[i] = d48_global
        elif key2 in gh_sum.index:
            ghm[i] = gh_sum.loc[key2] / gh_cnt.loc[key2]
        else:
            ghm[i] = d48_global

        # (g,h) lookup
        if key2 in gh_sum.index:
            s, c = gh_sum.loc[key2], gh_cnt.loc[key2]
            if excl:
                s -= demand_arr[i]; c -= 1
            gh[i] = s / c if c > 0 else d48_global
        else:
            gh[i] = d48_global

        # (g) lookup
        if key1 in g_sum.index:
            s, c = g_sum.loc[key1], g_cnt.loc[key1]
            if excl:
                s -= demand_arr[i]; c -= 1
            g[i] = s / c if c > 0 else d48_global
        else:
            g[i] = d48_global

    return ghm, gh, g

print('Computing d48 lookups for train (with self-exclusion for day 48 rows)...')
tr_ghm, tr_gh, tr_g = lookup_d48(train_fe, exclude_self_day48=True)
train_fe['d48_lookup_ghm'] = tr_ghm
train_fe['d48_lookup_gh']  = tr_gh
train_fe['d48_lookup_g']   = tr_g

print('Computing d48 lookups for test (no self-exclusion)...')
te_ghm, te_gh, te_g = lookup_d48(test_fe, exclude_self_day48=False)
test_fe['d48_lookup_ghm'] = te_ghm
test_fe['d48_lookup_gh']  = te_gh
test_fe['d48_lookup_g']   = te_g

print(f'\nLookup feature stats (train):')
print(train_fe[['d48_lookup_ghm','d48_lookup_gh','d48_lookup_g']].describe().round(4))
print(f'\nLookup feature stats (test):')
print(test_fe[['d48_lookup_ghm','d48_lookup_gh','d48_lookup_g']].describe().round(4))

---
## Step 3: Day-49 Trajectory + Calibration

From day-49 train rows (hours 0/1/2), compute per-geohash:  
- `d49_h0/h1/h2_demand`: actual day-49 demand at hour h  
- `d49_h0/h1/h2_ratio`: ratio of day-49 to day-48 demand at hour h (the **calibration factor**)  
- `d49_trend`: `d49_h1 - d49_h0` (momentum)  
- `d49_avg_ratio`: average day-49/day-48 ratio across known hours (per-geohash calibration)  

For test rows in hours 2-13, the model uses these per-geohash ratios to extrapolate the calibration.

In [ ]:
d49 = train_fe[train_fe['day'] == 49]
d48 = train_fe[train_fe['day'] == 48]

# Per-geohash day-49 demand at each hour
d49_h0 = d49[d49['hour'] == 0].groupby('geohash')['demand'].mean().to_dict()
d49_h1 = d49[d49['hour'] == 1].groupby('geohash')['demand'].mean().to_dict()
d49_h2 = d49[d49['hour'] == 2].groupby('geohash')['demand'].mean().to_dict()

# Per-geohash day-48 demand at hours 0/1/2 (for ratio computation)
d48_h0 = d48[d48['hour'] == 0].groupby('geohash')['demand'].mean().to_dict()
d48_h1 = d48[d48['hour'] == 1].groupby('geohash')['demand'].mean().to_dict()
d48_h2 = d48[d48['hour'] == 2].groupby('geohash')['demand'].mean().to_dict()

# Global fallback ratios
global_ratio_h0 = 2.13; global_ratio_h1 = 1.43; global_ratio_h2 = 1.22

def add_d49_features(df):
    fb = df['d48_lookup_g'].values
    gh_arr = df['geohash'].values
    n = len(df)

    d49h0 = np.empty(n); d49h1 = np.empty(n); d49h2 = np.empty(n)
    r_h0  = np.empty(n); r_h1  = np.empty(n); r_h2  = np.empty(n)

    for i in range(n):
        g = gh_arr[i]
        # demand levels (fallback: day 48 geohash mean)
        d49h0[i] = d49_h0.get(g, fb[i])
        d49h1[i] = d49_h1.get(g, fb[i])
        d49h2[i] = d49_h2.get(g, fb[i])
        # ratios at each hour (use d48 hour-specific level as denominator; fallback to global ratio)
        b0 = d48_h0.get(g); b1 = d48_h1.get(g); b2 = d48_h2.get(g)
        r_h0[i] = (d49_h0[g] / b0) if (g in d49_h0 and b0 and b0 > 1e-6) else global_ratio_h0
        r_h1[i] = (d49_h1[g] / b1) if (g in d49_h1 and b1 and b1 > 1e-6) else global_ratio_h1
        r_h2[i] = (d49_h2[g] / b2) if (g in d49_h2 and b2 and b2 > 1e-6) else global_ratio_h2

    df['d49_h0_demand'] = d49h0
    df['d49_h1_demand'] = d49h1
    df['d49_h2_demand'] = d49h2
    df['d49_trend']     = d49h1 - d49h0
    df['d49_h0_ratio']  = r_h0
    df['d49_h1_ratio']  = r_h1
    df['d49_h2_ratio']  = r_h2
    df['d49_avg_ratio'] = (r_h0 + r_h1 + r_h2) / 3
    # Direct calibrated baseline: d48_lookup_ghm * predicted_ratio_at_this_hour
    # Since we don't know ratio at hour h (h in 3..13), use d49_h2_ratio as the best available
    # (closest known hour). For h <= 2, use the specific hour ratio.
    h_arr = df['hour'].values
    cal_ratio = np.where(h_arr == 0, r_h0,
                np.where(h_arr == 1, r_h1, r_h2))
    df['calibrated_baseline'] = df['d48_lookup_ghm'].values * cal_ratio

    return df

train_fe = add_d49_features(train_fe)
test_fe  = add_d49_features(test_fe)

print(f'Train FE shape: {train_fe.shape}  |  Test FE shape: {test_fe.shape}')
print('\nCalibration features (test):')
print(test_fe[['d49_h0_ratio','d49_h1_ratio','d49_h2_ratio','d49_avg_ratio',
               'calibrated_baseline']].describe().round(4))

---
## Step 4: Target Encodings (α=10, full train)

In [ ]:
global_mean = float(train_fe['demand'].mean())
alpha = 10

def smooth_encode(group_cols, col_name):
    stats = train_fe.groupby(group_cols)['demand'].agg(['mean','count'])
    stats['smooth'] = (stats['count']*stats['mean'] + alpha*global_mean) / (stats['count'] + alpha)
    d = stats['smooth'].to_dict()
    if isinstance(group_cols, list):
        tr_keys = list(zip(*[train_fe[c] for c in group_cols]))
        te_keys = list(zip(*[test_fe[c] for c in group_cols]))
    else:
        tr_keys = train_fe[group_cols].tolist()
        te_keys = test_fe[group_cols].tolist()
    train_fe[col_name] = [d.get(k, global_mean) for k in tr_keys]
    test_fe[col_name]  = [d.get(k, global_mean) for k in te_keys]

smooth_encode('geohash',                  'geohash_mean_demand')
smooth_encode(['geohash', 'hour'],         'geohash_hour_mean_demand')
smooth_encode('geohash_4',                 'geohash_4_mean_demand')
smooth_encode(['geohash_4', 'hour'],       'geohash_4_hour_mean_demand')
smooth_encode('geohash_5',                 'geohash_5_mean_demand')

print('Target encodings added.')
print(f'Train columns: {len(train_fe.columns)}')

---
## Step 5: Feature Columns

In [ ]:
EXCLUDE = {'Index', 'geohash', 'timestamp', 'demand', 'RoadType', 'Weather',
           'geohash_4', 'geohash_5'}
feature_cols = [c for c in train_fe.columns if c not in EXCLUDE]
print(f'Total features: {len(feature_cols)}')
for i, c in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {c}')

In [ ]:
X_all = train_fe[feature_cols].values.astype(np.float32)
y_all = train_fe['demand'].values.astype(np.float32)
nan_cnt = int(np.isnan(X_all).sum())
if nan_cnt > 0:
    X_all = np.nan_to_num(X_all, nan=0.0)
    print(f'Replaced {nan_cnt} NaN in X_all')
print(f'X_all: {X_all.shape}  |  y range: [{y_all.min():.4f}, {y_all.max():.4f}]')

---
## Step 6: Validation — train day 48, val day 49

Mirrors the train→test temporal shift. Reporting only — final model uses full train.

In [ ]:
is_d48 = (train_fe['day'] == 48).values
tr_idx = np.where( is_d48)[0]
va_idx = np.where(~is_d48)[0]
X_train, y_train = X_all[tr_idx], y_all[tr_idx]
X_val,   y_val   = X_all[va_idx], y_all[va_idx]
print(f'Train (day 48): {X_train.shape}  |  Val (day 49): {X_val.shape}')

# Quick sanity: how does the raw calibrated_baseline alone perform on val?
calib_idx = feature_cols.index('calibrated_baseline')
r2_raw = r2_score(y_val, X_val[:, calib_idx])
print(f'Raw calibrated_baseline R^2 on val: {r2_raw:.4f}  ({max(0,r2_raw*100):.2f})')

---
## Step 7: Train LightGBM

In [ ]:
PARAMS = dict(
    learning_rate     = 0.05,
    num_leaves        = 255,
    min_child_samples = 30,
    subsample         = 0.85,
    subsample_freq    = 1,
    colsample_bytree  = 0.80,
    reg_alpha         = 0.3,
    reg_lambda        = 1.0,
    n_estimators      = 3000,
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

val_model = LGBMRegressor(**PARAMS)
val_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb_early_stop(50, verbose=False), lgb_log(200)]
)

train_r2 = r2_score(y_train, val_model.predict(X_train))
val_r2   = r2_score(y_val,   val_model.predict(X_val))
best_iter = val_model.best_iteration_

print(f'\nTrain R^2 = {train_r2:.4f}  |  Val R^2 = {val_r2:.4f}  |  Gap = {train_r2 - val_r2:.4f}')
print(f'Score est = {max(0, val_r2*100):.2f}  |  Best iter: {best_iter}')

imp = pd.DataFrame({'feature': feature_cols,
                     'importance': val_model.feature_importances_}
                   ).sort_values('importance', ascending=False)
print('\nTop 20 features:')
print(imp.head(20).to_string(index=False))

---
## Step 8: Retrain on Full Train + Predict Test

In [ ]:
final_iters = int(best_iter * 1.15) + 1
final_params = {k: v for k, v in PARAMS.items() if k != 'n_estimators'}
final_params['n_estimators'] = final_iters

print(f'Retraining on {X_all.shape[0]} rows with {final_iters} trees...')
final_model = LGBMRegressor(**final_params)
final_model.fit(X_all, y_all)

X_test = test_fe[feature_cols].values.astype(np.float32)
nan_te = int(np.isnan(X_test).sum())
if nan_te > 0:
    X_test = np.nan_to_num(X_test, nan=0.0)
    print(f'Replaced {nan_te} NaN in X_test')

final_preds = final_model.predict(X_test)
final_preds = np.clip(final_preds, 0.0, 1.0)

print(f'\nPrediction stats:')
print(f'  min  = {final_preds.min():.4f}')
print(f'  max  = {final_preds.max():.4f}')
print(f'  mean = {final_preds.mean():.4f}  (train mean: {y_all.mean():.4f})')
print(f'  std  = {final_preds.std():.4f}   (train std:  {y_all.std():.4f})')
print(f'  zeros: {100*(final_preds==0).mean():.1f}%')

In [ ]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': final_preds})
assert submission.shape == (41778, 2), f'Shape mismatch: {submission.shape}'
submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved  |  shape: {submission.shape}')
print('\nFirst 5 rows:')
print(submission.head())
print('\nDescribe:')
print(submission['demand'].describe())

---
## Step 9: Final Report

In [ ]:
print('=' * 70)
print('  GRIDLOCK HACKATHON 2.0 - v7 (lookup + calibration)')
print('=' * 70)
print(f'Validation (day 48 -> day 49):')
print(f'  Train R^2 = {train_r2:.4f}')
print(f'  Val   R^2 = {val_r2:.4f}  (score = {max(0,val_r2*100):.2f})')
print(f'  Gap       = {train_r2 - val_r2:.4f}')
print(f'  Raw calibrated_baseline R^2: {r2_raw:.4f}')
print(f'  Best iter (val): {best_iter}')

print(f'\nFinal submission model:')
print(f'  Trained on: {X_all.shape[0]} rows  |  Trees: {final_iters}')
print(f'  Features:   {len(feature_cols)}')

print(f'\nTop 10 features:')
for _, r in imp.head(10).iterrows():
    print(f'  {r["feature"]:<35s} {r["importance"]:.2f}')

print(f'\nSubmission stats:')
print(f'  min = {final_preds.min():.4f}  max = {final_preds.max():.4f}')
print(f'  mean = {final_preds.mean():.4f}  std = {final_preds.std():.4f}')
print(f'\nSubmission: submission.csv  shape={submission.shape}')
print('=' * 70)